# Time-Series Analysis: Monthly Default Trends & Macro Forecasting
**Issues: [R2-MX-01] #13 · [R2-MX-02] #14**

This notebook covers two tasks:
- **Issue #13 [R2-MX-01]** — Time-series data preparation: compute monthly default trends and reshape macroeconomic indicators.
- **Issue #14 [R2-MX-02]** — Time-series forecasting and evaluation using ARIMA/SARIMA with diagnostic plots and MAE/RMSE metrics.

---
**Data Sources:**
- `data/macroeconomic_indicators.csv` — monthly macro data (2023-01 to 2025-12)
- `data/borrower_profiles.csv` — borrower-level default status
- `data/monthly_transactions.csv` — per-borrower monthly transaction records

---
## Part 1 — Issue #13: Time-Series Data Preparation
### Step 1: Imports & Setup

In [ ]:
import sys, os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

sys.path.insert(0, os.path.dirname(os.getcwd()))

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d2e',
    'axes.edgecolor':   '#3a3f5c',
    'axes.labelcolor':  '#e0e0f0',
    'xtick.color':      '#a0a8d0',
    'ytick.color':      '#a0a8d0',
    'text.color':       '#e0e0f0',
    'grid.color':       '#2a2f4a',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'font.family':      'sans-serif',
    'figure.dpi':       120,
})

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')
PROC_DIR = os.path.join(DATA_DIR, 'processed')
print(f'Data dir: {DATA_DIR}')

### Step 2: Load & Reshape Macroeconomic Indicators

In [ ]:
macro = pd.read_csv(os.path.join(DATA_DIR, 'macroeconomic_indicators.csv'))
macro['date'] = pd.to_datetime(macro['date'])
macro = macro.sort_values('date').reset_index(drop=True)
macro.set_index('date', inplace=True)
macro.index.freq = 'MS'

print(f'Macro shape: {macro.shape}')
print(f'Date range : {macro.index.min().date()} to {macro.index.max().date()}')
macro.head()

### Step 3: Compute Monthly Default Trends from Borrower + Transaction Data

In [ ]:
bp = pd.read_csv(
    os.path.join(DATA_DIR, 'borrower_profiles.csv'),
    usecols=['borrower_id', 'loan_default_status']
)
tx = pd.read_csv(
    os.path.join(DATA_DIR, 'monthly_transactions.csv'),
    usecols=['borrower_id', 'month_label']
)
tx_default = tx.merge(bp, on='borrower_id', how='left')

monthly_defaults = (
    tx_default
    .groupby('month_label')
    .agg(total_borrowers=('borrower_id', 'nunique'),
         defaulters=('loan_default_status', 'sum'))
    .reset_index()
)
monthly_defaults['computed_default_rate'] = (
    monthly_defaults['defaulters'] / monthly_defaults['total_borrowers']
)
monthly_defaults['date'] = pd.to_datetime(monthly_defaults['month_label'])
monthly_defaults = monthly_defaults.sort_values('date').set_index('date')
monthly_defaults.index.freq = 'MS'

print(f'Monthly default trend shape: {monthly_defaults.shape}')
monthly_defaults.head(10)

### Step 4: Merge Computed Default Rate with Macro Indicators

In [ ]:
ts_df = macro.join(
    monthly_defaults[['computed_default_rate', 'total_borrowers', 'defaulters']],
    how='left'
)
ts_df['default_rate'] = ts_df['portfolio_default_rate']

print(f'Final time-series dataset shape: {ts_df.shape}')
ts_df.round(4)

### Step 5: Correlation Heatmap — Macro Indicators vs Default Rate

In [ ]:
corr_cols = ['cpi_inflation_rate', 'benchmark_interest_rate',
             'unemployment_rate', 'consumer_confidence_index', 'default_rate']

fig, ax = plt.subplots(figsize=(8, 6))
corr_matrix = ts_df[corr_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Macro Indicator Correlation Matrix', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

### Step 6: Plot Monthly Default Rate Trend + Macro Overlays

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)
fig.suptitle('Monthly Default Rate & Macroeconomic Indicators (2023-2025)',
             fontsize=15, color='#e0e0f0', y=1.01)

axes[0].plot(ts_df.index, ts_df['default_rate'], color='#7c83fd',
             linewidth=2.5, marker='o', markersize=4, label='Portfolio Default Rate')
axes[0].fill_between(ts_df.index, ts_df['default_rate'], alpha=0.2, color='#7c83fd')
axes[0].set_ylabel('Default Rate')
axes[0].legend(loc='upper left', framealpha=0.3)
axes[0].grid(True)

axes[1].plot(ts_df.index, ts_df['benchmark_interest_rate'], color='#fd7c83', linewidth=2, label='Benchmark Rate')
axes[1].plot(ts_df.index, ts_df['cpi_inflation_rate'], color='#ffd700', linewidth=2, linestyle='--', label='CPI Inflation')
axes[1].set_ylabel('Rate (%)')
axes[1].legend(loc='upper left', framealpha=0.3)
axes[1].grid(True)

ax3b = axes[2].twinx()
axes[2].plot(ts_df.index, ts_df['unemployment_rate'], color='#83fdc5', linewidth=2, label='Unemployment')
ax3b.plot(ts_df.index, ts_df['consumer_confidence_index'], color='#ff9f68', linewidth=2, linestyle=':', label='Confidence Index')
axes[2].set_ylabel('Unemployment (%)')
ax3b.set_ylabel('Confidence Index', color='#ff9f68')
axes[2].legend(loc='upper left', framealpha=0.3)
ax3b.legend(loc='upper right', framealpha=0.3)
axes[2].grid(True)
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%b-%Y'))
axes[2].xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()

### Step 7: Save Prepared Time-Series Dataset

In [ ]:
ts_df.to_csv(os.path.join(PROC_DIR, 'time_series_prepared.csv'))
print(f'Saved time_series_prepared.csv shape: {ts_df.shape}')
ts_df[['default_rate', 'cpi_inflation_rate', 'benchmark_interest_rate',
        'unemployment_rate', 'consumer_confidence_index']].describe().round(4)

---
## Part 2 — Issue #14: Time-Series Forecasting & Evaluation
### Step 8: Stationarity Tests (ADF & KPSS)

In [ ]:
series = ts_df['default_rate'].dropna()

def run_stationarity_tests(s, label='Series'):
    print(f'Stationarity Tests: {label}')
    adf_result = adfuller(s, autolag='AIC')
    print(f'  ADF Stat={adf_result[0]:.4f}  p={adf_result[1]:.4f}  -> {"Stationary" if adf_result[1] < 0.05 else "Non-stationary"}')
    kpss_result = kpss(s, regression='c', nlags='auto')
    print(f'  KPSS Stat={kpss_result[0]:.4f}  p={kpss_result[1]:.4f}  -> {"Stationary" if kpss_result[1] > 0.05 else "Non-stationary"}')

run_stationarity_tests(series, 'Portfolio Default Rate (original)')
series_diff = series.diff().dropna()
run_stationarity_tests(series_diff, 'Portfolio Default Rate (1st difference)')

### Step 9: ACF & PACF Plots (Order Selection)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('ACF & PACF — Original and Differenced Series', fontsize=14, color='#e0e0f0')

plot_acf(series,       lags=20, ax=axes[0, 0], color='#7c83fd', title='ACF - Original')
plot_pacf(series,      lags=15, ax=axes[0, 1], color='#fd7c83', title='PACF - Original', method='ywm')
plot_acf(series_diff,  lags=20, ax=axes[1, 0], color='#83fdc5', title='ACF - 1st Difference')
plot_pacf(series_diff, lags=15, ax=axes[1, 1], color='#ffd700', title='PACF - 1st Difference', method='ywm')

for ax in axes.flat:
    ax.set_facecolor('#1a1d2e')
    ax.tick_params(colors='#a0a8d0')
    ax.title.set_color('#e0e0f0')

plt.tight_layout()
plt.show()

### Step 10: Train / Test Split (80 / 20)

In [ ]:
n = len(series)
train_size = int(n * 0.80)
train = series.iloc[:train_size]
test  = series.iloc[train_size:]

print(f'Total: {n}  |  Train: {len(train)} ({train.index[0].date()} to {train.index[-1].date()})  |  Test: {len(test)} ({test.index[0].date()} to {test.index[-1].date()})')

### Step 11: ARIMA(1,1,1) — Fit & Forecast

In [ ]:
arima_model = ARIMA(train, order=(1, 1, 1))
arima_fit   = arima_model.fit()
print(arima_fit.summary())

arima_forecast = arima_fit.forecast(steps=len(test))
arima_forecast.index = test.index

arima_mae  = mean_absolute_error(test, arima_forecast)
arima_rmse = np.sqrt(mean_squared_error(test, arima_forecast))
print(f'ARIMA(1,1,1)  MAE={arima_mae:.6f}  RMSE={arima_rmse:.6f}')

### Step 12: SARIMA(1,1,1)(1,0,1,12) — Fit & Forecast

In [ ]:
sarima_model = SARIMAX(
    train, order=(1, 1, 1), seasonal_order=(1, 0, 1, 12),
    enforce_stationarity=False, enforce_invertibility=False
)
sarima_fit = sarima_model.fit(disp=False)
print(sarima_fit.summary())

sarima_forecast = sarima_fit.forecast(steps=len(test))
sarima_forecast.index = test.index

sarima_mae  = mean_absolute_error(test, sarima_forecast)
sarima_rmse = np.sqrt(mean_squared_error(test, sarima_forecast))
print(f'SARIMA(1,1,1)(1,0,1,12)  MAE={sarima_mae:.6f}  RMSE={sarima_rmse:.6f}')

### Step 13: Forecast Plot — Actual vs ARIMA vs SARIMA

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(train.index, train, color='#7c83fd', linewidth=2.5, marker='o', markersize=4, label='Train (Actual)')
ax.plot(test.index,  test,  color='#83fdc5', linewidth=2.5, marker='o', markersize=4, label='Test (Actual)')
ax.plot(arima_forecast.index,  arima_forecast,  color='#fd7c83', linewidth=2, linestyle='--',
        marker='x', markersize=6, label=f'ARIMA(1,1,1)  MAE={arima_mae:.4f}  RMSE={arima_rmse:.4f}')
ax.plot(sarima_forecast.index, sarima_forecast, color='#ffd700', linewidth=2, linestyle=':',
        marker='D', markersize=5, label=f'SARIMA(1,1,1)(1,0,1,12)  MAE={sarima_mae:.4f}  RMSE={sarima_rmse:.4f}')
ax.axvspan(test.index[0], test.index[-1], alpha=0.08, color='#83fdc5', label='Test Period')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax.set_title('Portfolio Default Rate: Actual vs ARIMA/SARIMA Forecasts', fontsize=14, color='#e0e0f0', pad=12)
ax.set_ylabel('Default Rate')
ax.legend(framealpha=0.3, fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()

### Step 14: Residual Diagnostics

In [ ]:
residuals   = sarima_fit.resid
fitted_vals = sarima_fit.fittedvalues

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('SARIMA Residual Diagnostics', fontsize=14, color='#e0e0f0')

axes[0,0].plot(residuals.index, residuals, color='#fd7c83', linewidth=1.5)
axes[0,0].axhline(0, color='#7c83fd', linestyle='--')
axes[0,0].set_title('Residuals over Time')

axes[0,1].hist(residuals, bins=15, color='#7c83fd', alpha=0.8, edgecolor='#3a3f5c')
axes[0,1].set_title('Residual Distribution')

plot_acf(residuals, lags=15, ax=axes[1,0], color='#83fdc5', title='ACF of Residuals')

axes[1,1].scatter(train, fitted_vals, color='#ffd700', alpha=0.7, s=30)
lims = [min(train.min(), fitted_vals.min())-0.001, max(train.max(), fitted_vals.max())+0.001]
axes[1,1].plot(lims, lims, color='#fd7c83', linewidth=1.5, linestyle='--')
axes[1,1].set_title('Fitted vs Actual (Train)')
axes[1,1].set_xlabel('Actual')
axes[1,1].set_ylabel('Fitted')

for ax in axes.flat:
    ax.set_facecolor('#1a1d2e')
    ax.tick_params(colors='#a0a8d0')
    ax.title.set_color('#e0e0f0')

plt.tight_layout()
plt.show()

### Step 15: Performance Summary Table

In [ ]:
results = pd.DataFrame({
    'Model': ['ARIMA(1,1,1)', 'SARIMA(1,1,1)(1,0,1,12)'],
    'MAE':   [arima_mae,  sarima_mae],
    'RMSE':  [arima_rmse, sarima_rmse],
    'AIC':   [arima_fit.aic,  sarima_fit.aic],
    'BIC':   [arima_fit.bic,  sarima_fit.bic],
}).set_index('Model')

print('Model Evaluation - Test Set Performance')
print(results.round(6).to_string())
print(f'Best MAE: {results["MAE"].idxmin()}  |  Best RMSE: {results["RMSE"].idxmin()}')

### Step 16: 3-Month Future Forecast with Confidence Intervals

In [ ]:
final_model = SARIMAX(
    series, order=(1, 1, 1), seasonal_order=(1, 0, 1, 12),
    enforce_stationarity=False, enforce_invertibility=False
).fit(disp=False)

FUTURE_STEPS = 3
future_forecast = final_model.get_forecast(steps=FUTURE_STEPS)
future_mean = future_forecast.predicted_mean
future_ci   = future_forecast.conf_int(alpha=0.20)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(series.index, series, color='#7c83fd', linewidth=2.5, marker='o', markersize=4, label='Historical Default Rate')
ax.plot(future_mean.index, future_mean, color='#ffd700', linewidth=2.5, linestyle='--', marker='D', markersize=7,
        label=f'Forecast ({FUTURE_STEPS}-month ahead)')
ax.fill_between(future_mean.index, future_ci.iloc[:, 0], future_ci.iloc[:, 1],
                color='#ffd700', alpha=0.2, label='80% Confidence Interval')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax.set_title(f'SARIMA: {FUTURE_STEPS}-Month Portfolio Default Rate Forecast', fontsize=14, color='#e0e0f0', pad=12)
ax.set_ylabel('Default Rate')
ax.legend(framealpha=0.3)
ax.grid(True)
plt.tight_layout()
plt.show()

print('Future Forecast:')
print(pd.DataFrame({'Forecast': future_mean, 'CI_Lower': future_ci.iloc[:,0], 'CI_Upper': future_ci.iloc[:,1]}).round(6))

---
## Summary

### Issue #13 - Time-Series Data Preparation
| Step | Description |
|------|-------------|
| 1 | Loaded and datetime-indexed `macroeconomic_indicators.csv` with MS frequency |
| 2 | Joined borrower_profiles with monthly_transactions to compute computed_default_rate per month |
| 3 | Merged macro indicators with default trends into `time_series_prepared.csv` |
| 4 | Verified correlation of macro features with the target (default rate) |

### Issue #14 - Time-Series Forecasting & Evaluation
| Step | Description |
|------|-------------|
| 1 | ADF and KPSS stationarity tests; 1st differencing confirmed stationarity |
| 2 | ACF/PACF plots used for ARIMA order selection |
| 3 | ARIMA(1,1,1) and SARIMA(1,1,1)(1,0,1,12) fitted and evaluated |
| 4 | Test metrics: MAE and RMSE reported for both models |
| 5 | Residual diagnostics: time plot, histogram, ACF, fitted vs actual |
| 6 | 3-month future forecast generated with 80% confidence intervals |